# Task A — SOLUTION NOTEBOOK
> **BAFAD Accelerated Research Track · Fall 2026 · INSTRUCTOR USE ONLY**

This file contains complete solutions and sample written answers. Do not distribute to students.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

NAVY  = '#021B3A'
BLUE  = '#0EA5E9'
plt.rcParams.update({'figure.dpi': 100, 'axes.spines.top': False, 'axes.spines.right': False})

df = pd.read_csv('../data/smap_sample.csv')
print(f'Loaded: {df.shape[0]} rows × {df.shape[1]} columns')

## 2 · First Look at the Data (2 pts)

In [ ]:
df.info()
df.describe()

**Sample answer (full marks):**

> The dataset has 500 rows and 27 columns: a `timestamp` column, 25 telemetry channels named `chan_00` through `chan_24`, and a binary `anomaly` column (0 = normal, 1 = anomaly). There are no missing values. All channel values are continuous floating-point numbers, and the `anomaly` column confirms this is a labelled dataset suitable for supervised or semi-supervised anomaly detection.

## 3 · Anomaly Prevalence (3 pts)

In [ ]:
n_anomaly = df['anomaly'].sum()
pct       = df['anomaly'].mean() * 100
print(f'{n_anomaly} of {len(df)} timesteps are anomalies ({pct:.1f}%).')
# Expected output: 24 of 500 timesteps are anomalies (4.8%).

**Grading note:** Award full marks if `n_anomaly`, `pct`, and the f-string format are all correct. Common error: forgetting `* 100` on pct, or using `len(df['anomaly'])` instead of `len(df)` (same result here, but conceptually sloppy).

## 4 · Time-Series Visualisation (5 pts)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

# Plot full line in NAVY
ax.plot(df['timestamp'], df['chan_00'], color=NAVY, linewidth=0.8, label='Normal')

# Overlay anomaly points in red
anomaly_mask = df['anomaly'] == 1
ax.scatter(df.loc[anomaly_mask, 'timestamp'],
           df.loc[anomaly_mask, 'chan_00'],
           color='red', s=8, zorder=5, label='Anomaly')

ax.set_xlabel('Timestamp (seconds)')
ax.set_ylabel('Channel 00 Value')
ax.set_title('SMAP Channel 00 — Telemetry with Anomalies Highlighted')
ax.legend()
plt.tight_layout()
plt.show()

**Grading note (5 pts):**
- 2 pts: line plotted with NAVY colour
- 2 pts: anomaly markers correctly filtered and plotted in red
- 1 pt: axis labels and title present

Common errors: plotting anomalies on the wrong axis; using `df['anomaly'] == True` (works, but penalise nothing); omitting legend; using `'r.'` without `s=8` (fine, still award full marks if markers are visible).

## 5 · Multi-Channel Heatmap (5 pts)

In [ ]:
chan_cols    = df.filter(like='chan_').columns.tolist()
data_matrix = df[chan_cols].values.T  # shape: (25 channels, 500 timesteps)

fig, ax = plt.subplots(figsize=(14, 5))

im = ax.imshow(data_matrix, aspect='auto', cmap='RdBu_r', vmin=-2, vmax=2)
plt.colorbar(im, ax=ax, label='Normalised Value')

# Draw red vertical lines at anomaly timesteps
for idx in df.index[df['anomaly'] == 1]:
    ax.axvline(x=idx, color='red', linewidth=0.5, alpha=0.7)

ax.set_xlabel('Timestep Index')
ax.set_ylabel('Channel')
ax.set_yticks(range(len(chan_cols)))
ax.set_yticklabels(chan_cols, fontsize=6)
ax.set_title('All 25 SMAP Channels — Heatmap (red = anomaly timestep)')
plt.tight_layout()
plt.show()

**Grading note (5 pts):**
- 2 pts: `imshow` with correct `aspect='auto'`, `cmap='RdBu_r'`, `vmin=-2, vmax=2`
- 1 pt: colourbar present and labelled
- 2 pts: anomaly vertical lines drawn (colour and width are flexible)

Common errors: forgetting `.T` transpose (channels on x-axis instead of y-axis); using `df.index` vs `range(len(df))` (same here since index is 0-based); not using `aspect='auto'` (image looks square).

## 6 · Distribution of Channel Values (5 pts)

In [ ]:
normal  = df[df['anomaly'] == 0]['chan_00']
anomaly = df[df['anomaly'] == 1]['chan_00']

fig, ax = plt.subplots(figsize=(8, 4))

ax.hist(normal,  bins=30, color=NAVY, alpha=0.6, label=f'Normal (n={len(normal)})',  density=True)
ax.hist(anomaly, bins=15, color=BLUE, alpha=0.6, label=f'Anomaly (n={len(anomaly)})', density=True)

ax.set_xlabel('chan_00 Value')
ax.set_ylabel('Density')
ax.set_title('Channel 00: Normal vs Anomaly Distribution')
ax.legend()
plt.tight_layout()
plt.show()

**Sample observation (full marks):**

> The normal timesteps follow a roughly sinusoidal distribution centred near zero, with most values falling between approximately −1 and +1. The anomaly distribution is shifted toward higher positive values and is more dispersed, consistent with spike-type anomalies injected above the normal signal baseline. The separation between the two distributions suggests that a threshold on channel value — or, more robustly, on reconstruction error from an autoencoder trained only on normal data — could serve as an effective anomaly detector.

**Grading note (5 pts):**
- 2 pts: both histograms plotted with correct colours and alpha
- 1 pt: legend present
- 1 pt: density normalisation (either `density=True` or explicit, not required but best practice)
- 1 pt: written observation addresses the visible difference between distributions

Do not penalise students who omit `density=True`; the visual comparison is still valid.

## 7 · Reflection (2 pts)

**Sample reflection (full marks):**

> What surprised me most was how rare the anomalies actually are — only 24 out of 500 timesteps (4.8%). This class imbalance immediately explains why a naive classifier that always predicts 'normal' would be 95% accurate yet completely useless. The heatmap also revealed that anomalies affect multiple channels simultaneously, which makes single-channel thresholding unreliable and motivates the use of a multi-channel model like the BAFAD autoencoder.
>
> The EDA patterns guide the model design in two ways: first, training only on normal data (as the BAFAD autoencoder does) avoids the problem of having too few anomaly examples to learn from; second, the multi-channel view suggests that reconstruction error aggregated across all channels will be a more robust anomaly score than any per-channel statistic.

**Grading note (2 pts):** Award 2 pts for any response that (a) mentions the class imbalance or rarity of anomalies and (b) connects an observed EDA pattern to a modelling decision. Award 1 pt for a vague but good-faith attempt. Award 0 for a blank or single-sentence response.